# Profiler Run

Notebook para ejecutar corridas de profiling/training y guardar `trace` + `summary.json` + `events.jsonl`.

In [1]:
from pathlib import Path
import sys
import pandas as pd
from IPython.display import display

try:
    from profiler.profiler_tools import run_profiler, run_profiler_grad_acc, run_resource_sweep, run_resource_sweep_grad_acc, collect_saved_runs, build_core_summary
except ModuleNotFoundError:
    candidates = [Path.cwd(), Path.cwd().parent]
    for cand in candidates:
        if (cand / "profiler" / "profiler_tools.py").exists() and str(cand) not in sys.path:
            sys.path.insert(0, str(cand))
    from profiler.profiler_tools import run_profiler, run_profiler_grad_acc, run_resource_sweep, run_resource_sweep_grad_acc, collect_saved_runs, build_core_summary


def find_project_root():
    for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
        if (p / "src").exists() and (p / "profiler").exists():
            return p
    raise RuntimeError("No se encontro project root (esperado: carpetas src/ y profiler/).")


PROJECT_ROOT = find_project_root()
PROFILER_DIR = PROJECT_ROOT / "profiler"
LOG_ROOT = PROFILER_DIR / "logs" / "diffusion_profiler_light_batch"

print(f"PROJECT_ROOT={PROJECT_ROOT}")
print(f"LOG_ROOT={LOG_ROOT}")

PROJECT_ROOT=/home/gkulemeyer/Documents/Repos/RNADiffusion
LOG_ROOT=/home/gkulemeyer/Documents/Repos/RNADiffusion/profiler/logs/diffusion_profiler_light_batch


In [2]:
BASE_DATA_DIR = PROJECT_ROOT / "data" / "simfolds" / "simfolds_max128" / "joined"
sim = "sim90"
DATA_DIR = BASE_DATA_DIR / sim
DATA_DIR.mkdir(parents=True, exist_ok=True)

BASE_CONF = {
    "train_path": str(DATA_DIR / "train.csv"),
    "val_path": str(DATA_DIR / "valid.csv"),
    "log_path": str(LOG_ROOT),
    "batch_size": 4,
    "lr": 1e-3,
    "epochs": 1,
    "timesteps": 10,
    "num_workers": 2,
    "profile_batches": 4,
    "trace_wait": 0,
    "trace_warmup": 1,
    "trace_active": 12,
    "record_shapes": False,
    "profile_memory": True,
    "with_stack": False,
    "grad_accum_steps": 1,
    "append_timestamp": False,
    "use_amp": False,
    "amp_sampling": False,
    "sim_tag": sim,
    "note": f"resource profiler: {sim}. Archive II max 128.",
}

single_conf = dict(BASE_CONF)
single_conf["timesteps"] = 10
single_conf["batch_size"] = 4
single_conf["use_amp"] = False
single_conf["grad_accum_steps"] = 1
single_conf["run_name"] = f"{sim}_bs4_pb4_10ts_fp32"
single_conf["folder_name"] = single_conf["run_name"]

single_conf

{'train_path': '/home/gkulemeyer/Documents/Repos/RNADiffusion/data/simfolds/simfolds_max128/joined/sim90/train.csv',
 'val_path': '/home/gkulemeyer/Documents/Repos/RNADiffusion/data/simfolds/simfolds_max128/joined/sim90/valid.csv',
 'log_path': '/home/gkulemeyer/Documents/Repos/RNADiffusion/profiler/logs/diffusion_profiler_light_batch',
 'batch_size': 4,
 'lr': 0.001,
 'epochs': 1,
 'timesteps': 10,
 'num_workers': 2,
 'profile_batches': 4,
 'trace_wait': 0,
 'trace_warmup': 1,
 'trace_active': 12,
 'record_shapes': False,
 'profile_memory': True,
 'with_stack': False,
 'grad_accum_steps': 1,
 'append_timestamp': False,
 'use_amp': False,
 'amp_sampling': False,
 'sim_tag': 'sim90',
 'note': 'resource profiler: sim90. Archive II max 128.',
 'run_name': 'sim90_bs4_pb4_10ts_fp32',
 'folder_name': 'sim90_bs4_pb4_10ts_fp32'}

In [3]:
# Ejecutar corrida unica
# result = run_profiler(single_conf)
# result

# Ejecutar corrida unica con acumulacion de gradiente
# result_ga = run_profiler_grad_acc(single_conf, grad_accum_steps=4)
# result_ga


In [7]:
# Sweep batch_size x grad_accum_steps (configuracion solicitada)
sweep_ga_df = run_resource_sweep_grad_acc(
    BASE_CONF,
    timesteps_list=[60, 70, 80, 90, 100],
    batch_sizes=[1],
    grad_accum_steps_list=[4, 8, 16],
    amp_modes=[False, True],
    output_log_path=LOG_ROOT,
    run_name_prefix="batch_ga_sweep",
)

display(sweep_ga_df.sort_values(["timesteps", "batch_size", "grad_accum_steps", "use_amp"]))


STAGE:2026-03-03 22:26:14 4122452:4122452 ActivityProfilerController.cpp:312] Completed Stage: Warm Up
STAGE:2026-03-03 22:26:26 4122452:4122452 ActivityProfilerController.cpp:318] Completed Stage: Collection
STAGE:2026-03-03 22:26:27 4122452:4122452 ActivityProfilerController.cpp:322] Completed Stage: Post Processing
STAGE:2026-03-03 22:28:55 4122452:4122452 ActivityProfilerController.cpp:312] Completed Stage: Warm Up
STAGE:2026-03-03 22:29:08 4122452:4122452 ActivityProfilerController.cpp:318] Completed Stage: Collection
STAGE:2026-03-03 22:29:09 4122452:4122452 ActivityProfilerController.cpp:322] Completed Stage: Post Processing
STAGE:2026-03-03 22:31:58 4122452:4122452 ActivityProfilerController.cpp:312] Completed Stage: Warm Up
STAGE:2026-03-03 22:32:10 4122452:4122452 ActivityProfilerController.cpp:318] Completed Stage: Collection
STAGE:2026-03-03 22:32:13 4122452:4122452 ActivityProfilerController.cpp:322] Completed Stage: Post Processing
STAGE:2026-03-03 22:34:39 4122452:412245

,status,error,run_name,run_id,run_folder,run_dir,trace_dir,trace_path,timesteps,batch_size,...,train_step_alloc_delta_avg_mb_gpu,train_step_alloc_delta_peak_mb_gpu,train_step_alloc_traffic_mb_gpu,train_step_free_traffic_mb_gpu,train_step_alloc_avg_mb_cpu,train_step_alloc_peak_mb_cpu,train_step_alloc_delta_avg_mb_cpu,train_step_alloc_delta_peak_mb_cpu,train_step_alloc_traffic_mb_cpu,train_step_free_traffic_mb_cpu
0,ok,,batch_ga_sweep_sim90_bs1_ga4_pb4_60ts_fp32,batch_ga_sweep_sim90_bs1_ga4_pb4_60ts_fp32,batch_ga_sweep_sim90_bs1_ga4_pb4_60ts_fp32,/home/gkulemeyer/Documents/Repos/RNADiffusion/...,/home/gkulemeyer/Documents/Repos/RNADiffusion/...,/home/gkulemeyer/Documents/Repos/RNADiffusion/...,60,1,...,18.334255,31.163086,31.163086,10.387695,0.008420,0.008457,0.000287,0.000324,0.001251,0.000938
1,ok,,batch_ga_sweep_sim90_bs1_ga4_pb4_60ts_amp,batch_ga_sweep_sim90_bs1_ga4_pb4_60ts_amp,batch_ga_sweep_sim90_bs1_ga4_pb4_60ts_amp,/home/gkulemeyer/Documents/Repos/RNADiffusion/...,/home/gkulemeyer/Documents/Repos/RNADiffusion/...,/home/gkulemeyer/Documents/Repos/RNADiffusion/...,60,1,...,18.256078,32.429199,32.431641,11.250000,0.008733,0.008770,0.000287,0.000324,0.001251,0.000938
2,ok,,batch_ga_sweep_sim90_bs1_ga8_pb4_60ts_fp32,batch_ga_sweep_sim90_bs1_ga8_pb4_60ts_fp32,batch_ga_sweep_sim90_bs1_ga8_pb4_60ts_fp32,/home/gkulemeyer/Documents/Repos/RNADiffusion/...,/home/gkulemeyer/Documents/Repos/RNADiffusion/...,/home/gkulemeyer/Documents/Repos/RNADiffusion/...,60,1,...,18.334255,31.163086,31.163086,10.387695,0.009046,0.009083,0.000287,0.000324,0.001251,0.000938
3,ok,,batch_ga_sweep_sim90_bs1_ga8_pb4_60ts_amp,batch_ga_sweep_sim90_bs1_ga8_pb4_60ts_amp,batch_ga_sweep_sim90_bs1_ga8_pb4_60ts_amp,/home/gkulemeyer/Documents/Repos/RNADiffusion/...,/home/gkulemeyer/Documents/Repos/RNADiffusion/...,/home/gkulemeyer/Documents/Repos/RNADiffusion/...,60,1,...,19.174874,33.030762,33.033203,10.390625,0.009359,0.009396,0.000287,0.000324,0.001251,0.000938
4,ok,,batch_ga_sweep_sim90_bs1_ga16_pb4_60ts_fp32,batch_ga_sweep_sim90_bs1_ga16_pb4_60ts_fp32,batch_ga_sweep_sim90_bs1_ga16_pb4_60ts_fp32,/home/gkulemeyer/Documents/Repos/RNADiffusion/...,/home/gkulemeyer/Documents/Repos/RNADiffusion/...,/home/gkulemeyer/Documents/Repos/RNADiffusion/...,60,1,...,18.671146,31.569336,31.569336,10.387695,0.009671,0.009708,0.000287,0.000324,0.001251,0.000938
5,ok,,batch_ga_sweep_sim90_bs1_ga16_pb4_60ts_amp,batch_ga_sweep_sim90_bs1_ga16_pb4_60ts_amp,batch_ga_sweep_sim90_bs1_ga16_pb4_60ts_amp,/home/gkulemeyer/Documents/Repos/RNADiffusion/...,/home/gkulemeyer/Documents/Repos/RNADiffusion/...,/home/gkulemeyer/Documents/Repos/RNADiffusion/...,60,1,...,18.460053,32.300293,32.302734,10.775391,0.009984,0.010021,0.000287,0.000324,0.001251,0.000938
6,ok,,batch_ga_sweep_sim90_bs1_ga4_pb4_70ts_fp32,batch_ga_sweep_sim90_bs1_ga4_pb4_70ts_fp32,batch_ga_sweep_sim90_bs1_ga4_pb4_70ts_fp32,/home/gkulemeyer/Documents/Repos/RNADiffusion/...,/home/gkulemeyer/Documents/Repos/RNADiffusion/...,/home/gkulemeyer/Documents/Repos/RNADiffusion/...,70,1,...,19.265241,32.889648,32.889648,11.258789,0.010297,0.010334,0.000287,0.000324,0.001251,0.000938
7,ok,,batch_ga_sweep_sim90_bs1_ga4_pb4_70ts_amp,batch_ga_sweep_sim90_bs1_ga4_pb4_70ts_amp,batch_ga_sweep_sim90_bs1_ga4_pb4_70ts_amp,/home/gkulemeyer/Documents/Repos/RNADiffusion/...,/home/gkulemeyer/Documents/Repos/RNADiffusion/...,/home/gkulemeyer/Documents/Repos/RNADiffusion/...,70,1,...,0.000895,0.001953,0.002930,0.002930,NaN,NaN,NaN,NaN,NaN,NaN
8,ok,,batch_ga_sweep_sim90_bs1_ga8_pb4_70ts_fp32,batch_ga_sweep_sim90_bs1_ga8_pb4_70ts_fp32,batch_ga_sweep_sim90_bs1_ga8_pb4_70ts_fp32,/home/gkulemeyer/Documents/Repos/RNADiffusion/...,/home/gkulemeyer/Documents/Repos/RNADiffusion/...,/home/gkulemeyer/Documents/Repos/RNADiffusion/...,70,1,...,18.334255,31.163086,31.163086,10.387695,0.010610,0.010647,0.000287,0.000324,0.001251,0.000938
9,ok,,batch_ga_sweep_sim90_bs1_ga8_pb4_70ts_amp,batch_ga_sweep_sim90_bs1_ga8_pb4_70ts_amp,batch_ga_sweep_sim90_bs1_ga8_pb4_70ts_amp,

In [8]:
saved_df = collect_saved_runs(LOG_ROOT)
if len(saved_df) == 0:
    print("No se encontraron corridas en", LOG_ROOT)
else:
    core_df = build_core_summary(saved_df).sort_values(["timesteps", "use_amp"])
    display(core_df.head(20))

,run_name,status,timesteps,batch_size,use_amp,oom_events,trace_path,trace_size_mb,train_batch_ms,train_forward_ms,...,train_backward_alloc_traffic_mb_gpu,train_backward_free_traffic_mb_gpu,train_step_alloc_traffic_mb_gpu,train_step_free_traffic_mb_gpu,train_forward_alloc_traffic_mb_cpu,train_forward_free_traffic_mb_cpu,train_backward_alloc_traffic_mb_cpu,train_backward_free_traffic_mb_cpu,train_step_alloc_traffic_mb_cpu,train_step_free_traffic_mb_cpu
0,batch_ga_sweep_sim90_bs1_ga1_pb4_10ts_fp32,ok,10,1,False,0.0,/home/gkulemeyer/Documents/Repos/RNADiffusion/...,126.943350,304.657000,130.517000,...,11105.359863,11729.752767,10.642904,10.642904,0.001160,0.001152,0.000153,0.000160,0.000938,0.000938
1,batch_ga_sweep_sim90_bs1_ga2_pb4_10ts_fp32,ok,10,1,False,0.0,/home/gkulemeyer/Documents/Repos/RNADiffusion/...,119.311906,233.342000,105.050333,...,10516.226074,11084.727051,20.775391,10.387695,0.001160,0.001152,0.000153,0.000160,0.001095,0.000938
2,batch_ga_sweep_sim90_bs1_ga8_pb4_10ts_fp32,ok,10,1,False,0.0,/home/gkulemeyer/Documents/Repos/RNADiffusion/...,121.379138,300.359667,141.826667,...,9632.360840,10206.355306,31.163086,10.387695,0.001160,0.001152,0.000153,0.000160,0.001251,0.000938
3,batch_ga_sweep_sim90_bs4_ga1_pb4_10ts_fp32,ok,10,4,False,0.0,/home/gkulemeyer/Documents/Repos/RNADiffusion/...,119.977223,245.755000,110.517333,...,12113.355957,15095.801270,10.481445,10.481445,0.001160,0.001152,0.000153,0.000160,0.000938,0.000938
4,batch_ga_sweep_sim90_bs4_ga2_pb4_10ts_fp32,ok,10,4,False,0.0,/home/gkulemeyer/Documents/Repos/RNADiffusion/...,118.976442,306.180667,156.195667,...,14841.738770,18456.573242,21.509766,11.122070,0.001160,0.001152,0.000153,0.000160,0.001095,0.000938
5,batch_ga_sweep_sim90_bs4_ga8_pb4_10ts_fp32,ok,10,4,False,0.0,/home/gkulemeyer/Documents/Repos/RNADiffusion/...,115.108774,226.641333,107.157333,...,17735.467773,21515.707194,32.731445,11.122070,0.001160,0.001152,0.000153,0.000160,0.001251,0.000938
6,batch_ga_sweep_sim90_bs1_ga1_pb4_10ts_amp,ok,10,1,True,0.0,/home/gkulemeyer/Documents/Repos/RNADiffusion/...,150.929397,402.212333,177.473000,...,6376.860840,6994.212891,11.010417,11.010417,0.001160,0.001152,0.000153,0.000160,0.000938,0.000938
7,batch_ga_sweep_sim90_bs1_ga2_pb4_10ts_amp,ok,10,1,True,0.0,/home/gkulemeyer/Documents/Repos/RNADiffusion/...,148.617901,420.739333,200.912000,...,8106.294108,8619.881673,21.400391,10.798828,0.001160,0.001152,0.000153,0.000160,0.001095,0.000938
8,batch_ga_sweep_sim90_bs1_ga8_pb4_10ts_amp,ok,10,1,True,0.0,/home/gkulemeyer/Documents/Repos/RNADiffusion/...,140.530616,241.767333,94.895000,...,6479.643392,6880.775553,31.751953,10.390625,0.001160,0.001152,0.000153,0.000160,0.001251,0.000938
9,batch_ga_sweep_sim90_bs4_ga1_pb4_10ts_amp,ok,10,4,True,0.0,/home/gkulemeyer/Documents/Repos/RNADiffusion/...,146.540525,391.383333,187.465333,...,15394.493164,18718.732585,18.588216,11.243164,0.001160,0.001152,0.000153,0.000160,0.001043,0.000938
